# 🚩 RAG Agéntico para Señales de Riesgo en Contratación Pública

**Proyecto final — Sistema RAG en Google Colab con Qwen.**

Analiza un **contrato/licitación** y emite **señales de riesgo potenciales** (red flags) fundamentadas en la guía *OCP 2024 — Red Flags for Procurement*, citando la fuente y el dato del contrato. **No afirma corrupción**: toda salida requiere **revisión humana**.

## Arquitectura (RAG agéntico)
```
CONTRATO
  → router (familia: planeación / competencia / adjudicación / ejecución)
  → retrieval híbrido (BM25 + FAISS) top-20
  → reranker cross-encoder (bge-reranker-v2-m3) → top-5
  → generación Qwen2.5-3B-Instruct (grounded en los chunks)
  → verificación de grounding (soporte por frase) + citas por frase
  → REPORTE: señales de riesgo + severidad + citas + 'requiere revisión humana'
```

**Modelos (todos de HuggingFace):** `intfloat/multilingual-e5-base` (embeddings) · `BAAI/bge-reranker-v2-m3` (reranker) · `Qwen/Qwen2.5-3B-Instruct` (generación).

> **Antes de correr:** Runtime → *Change runtime type* → **GPU (T4)**. Edita `REPO_URL` en la celda 1.2 con tu repositorio de GitHub.

## 1. Instalación de librerías y repositorio

In [4]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA disponible: True
GPU: Tesla T4


In [ ]:
# 1.0 Token de Hugging Face (Colab Secrets o manual; nunca se imprime)
import os
from getpass import getpass
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if not hf_token:
    hf_token = getpass("HF_TOKEN no encontrado en Secrets. Pegalo o deja vacio: ")
os.environ["HF_TOKEN"] = hf_token or ""
print("HF_TOKEN cargado:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# 1.1 Dependencias (en Colab T4)
# 1.1 Dependencias principales del RAG
!pip -q install \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate \
  rank_bm25 \
  rouge-score \
  pymupdf \
  numpy \
  pandas \
  scikit-learn \
  nltk

In [ ]:
import requests
import faiss
import sentence_transformers
import transformers
import rank_bm25
import fitz  # pymupdf

print("requests:", requests.__version__)
print("faiss OK")
print("sentence-transformers OK")
print("transformers OK")
print("rank_bm25 OK")
print("pymupdf OK")

In [ ]:
# 1.2 Clonar el repo (rag_core, evals, datos). EDITA REPO_URL con tu GitHub.
import os, sys
REPO_URL = "https://github.com/MiguelAAR10/rag-redflags-colab.git"
REPO_DIR = "/content/" + REPO_URL.rstrip('/').split('/')[-1].replace('.git','')
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
# Mismas rutas que los runners: repo_root + repo_root/packages
for p in [REPO_DIR, os.path.join(REPO_DIR, 'packages')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print('Repo:', REPO_DIR)

## 2. Carga y exploración del dataset

**Fuente:** guía *OCP 2024 — Red Flags for Procurement* (PDF, ~100 pp). **Dominio:** integridad en contratación pública.

**Definición clave:** *documento* = **unidad documental lógica** del libro (cada indicador se parte en bloques `core` / `formula` / `example`; metodología por subsección), **no** un chunk arbitrario. **Dificultades:** layout a columnas, fórmulas en prosa, bilingüe EN/ES, sin OCR (texto nativo).

In [ ]:
# 2.1 Asegurar el PDF en data/raw/ (súbelo si no está)
from pathlib import Path
from rag_core.loaders import PDF_PATH
PDF_PATH = Path(PDF_PATH)
if not PDF_PATH.exists():
    PDF_PATH.parent.mkdir(parents=True, exist_ok=True)
    from google.colab import files
    print('Sube OCP2024-RedFlagProcurement-1.pdf:')
    up = files.upload()
    Path(list(up.keys())[0]).replace(PDF_PATH)
print('PDF:', PDF_PATH, '| existe:', PDF_PATH.exists())

In [ ]:
# 2.2 Parsear el PDF en unidades documentales lógicas
import json
from rag_core.loaders import load_pdf, generate_report
units = load_pdf()
report = generate_report(units)
print('Total unidades:', len(units))
print(json.dumps(report, indent=2, ensure_ascii=False)[:900])
print('\nEjemplo de unidad:')
{k: (v[:120] + '...' if isinstance(v, str) and len(v) > 120 else v) for k, v in units[0].items()}

## 3. Chunking + comparación de tamaños (con overlap)

**Tradeoff:** chunks pequeños → más precisión pero más fragmentación; grandes → más contexto pero dilución. El **overlap** preserva el contexto en las fronteras (que no se parta una fórmula). El chunking se aplica **sobre las unidades lógicas** y hereda su metadata.

In [ ]:
# 3.1 Comparar 3 configuraciones (size/overlap)

import json
from pathlib import Path
from rag_core.chunkers import compare_chunk_configs, chunk_units, save_chunks

# La función espera: (size, overlap, label)
configs = [
    (512, 64, "small"),
    (1024, 128, "medium"),
    (2048, 256, "large"),
]

comparison = compare_chunk_configs(units, configs)

print(json.dumps(comparison, indent=2, ensure_ascii=False))

# Configuración elegida: 1024/128
chunks = chunk_units(units, size=1024, overlap=128)
save_chunks(chunks, Path("data/processed/redflags_chunks.jsonl"))

print("\nChunks (1024/128):", len(chunks))
print("Guardado en: data/processed/redflags_chunks.jsonl")

## 4. Generación de embeddings

**Modelo:** `intfloat/multilingual-e5-base` (768-dim, multilingüe EN/ES). **Por qué:** el corpus está en inglés y los contratos pueden estar en español; e5 multilingüe lo cubre y rinde bien en MTEB. Los embeddings son vectores que representan el **significado** del texto → permiten búsqueda semántica (no solo por palabra clave).

In [ ]:
# 4.1 Embeddings normalizados (coseno via producto interno)
import numpy as np
from rag_core.embeddings import embed_texts
texts = [c['text'] for c in chunks]
vectors = embed_texts(texts)
print('Embeddings:', vectors.shape, '| norma del primero:', round(float(np.linalg.norm(vectors[0])), 4))

## 5. Indexación con FAISS

**Búsqueda vectorial:** dado un vector de consulta, FAISS encuentra los vectores más cercanos. **top-k:** los *k* chunks más similares. Elegimos **k=5** (suficiente contexto sin diluir; se compara en la evaluación). `IndexFlatIP` = búsqueda exacta por producto interno (coseno con vectores normalizados). **BONUS:** `HNSW` = grafo navegable, búsqueda aproximada O(log n) — útil si el dataset escala.

In [ ]:
# 5.1 Construir y guardar el índice FAISS (Flat) + mapping; BONUS HNSW
from pathlib import Path
from rag_core.indexing import build_index, save_index, build_mapping, save_mapping
index = build_index(vectors, use_hnsw=False)
save_index(index, Path('data/index/redflags_flatip.index'))
mapping = build_mapping(chunks)
save_mapping(mapping, Path('data/index/chunk_id_mapping.json'))
print('FAISS IndexFlatIP | ntotal:', index.ntotal, '| dim:', index.d)
# BONUS: HNSW
hnsw = build_index(vectors, use_hnsw=True)
print('FAISS HNSW        | ntotal:', hnsw.ntotal)

### Criterios: embeddings, FAISS top-k y HNSW

- **Embeddings**: cada chunk/consulta -> vector de 768 dim (e5-base) que representa su *significado*; normalizados -> similitud coseno = producto interno (IP).
- **Busqueda vectorial + top-k**: FAISS devuelve los **k** vectores mas cercanos a la consulta. Usamos **k=5** (suficiente contexto sin diluir; se compara en la evaluacion, seccion 9).
- **IndexFlatIP**: busqueda **exacta** O(N*d). Con N=299 es instantanea -> es la eleccion correcta aqui.
- **HNSW** (bonus): grafo navegable, busqueda **aproximada** O(log N). NO mejora con N pequeno; **gana mucho al escalar**. Lo demostramos honestamente abajo.

In [ ]:
# 5.2 Por que HNSW? Benchmark honesto Flat vs HNSW al crecer N
import numpy as np, time, faiss
def bench(n, d=768, q=100, k=5):
    rng = np.random.RandomState(0)
    xb = rng.randn(n, d).astype('float32'); faiss.normalize_L2(xb)
    xq = rng.randn(q, d).astype('float32'); faiss.normalize_L2(xq)
    flat = faiss.IndexFlatIP(d); flat.add(xb)
    t0 = time.time(); flat.search(xq, k); t_flat = (time.time()-t0)/q*1000
    hnsw = faiss.IndexHNSWFlat(d, 32); hnsw.add(xb)
    t0 = time.time(); hnsw.search(xq, k); t_hnsw = (time.time()-t0)/q*1000
    return t_flat, t_hnsw

print(f"{'N':>8} {'Flat ms/q':>12} {'HNSW ms/q':>12}")
for n in [299, 5000, 50000]:
    tf, th = bench(n)
    print(f"{n:>8} {tf:>12.3f} {th:>12.3f}")
print("\nN=299 (nuestro caso): ambos instantaneos -> Flat exacto es la eleccion.")
print("Al crecer N, HNSW (aproximado, O(log N)) gana claramente -> util si el corpus escala.")

## 6. Retrieval top-k (híbrido BM25 + FAISS + router de familias)

**BONUS Hybrid Search:** combinamos BM25 (léxico) + FAISS (semántico) vía *Reciprocal Rank Fusion*. El **router** clasifica la consulta en una familia de red flags para acotar/priorizar la búsqueda.

In [ ]:
# 6.1 Demo de retrieval híbrido + router
from rag_core.retrievers import load_chunks, build_bm25, route_family, hybrid_search
chunks = load_chunks()
bm25 = build_bm25(chunks)  # rank_bm25 instalado en Colab -> BM25 activo
q = 'single bidder with a very short submission period'
print('Familia(s) detectada(s):', route_family(q))
for r in hybrid_search(q, k=5, bm25_obj=bm25, chunks=chunks):
    print(round(r['score'], 3), '|', r.get('indicator_code'), '|', r.get('indicator_name'), '| p', r.get('page_start'))

## 7. Integración con el LLM (Qwen2.5-3B) + reranker

`agent.analyze` orquesta: retrieve híbrido top-20 → **reranker cross-encoder** top-5 → **Qwen2.5-3B-Instruct** genera observaciones fundamentadas SOLO en los chunks, con **lenguaje seguro** (señales de riesgo, requiere revisión humana).

In [ ]:
# 7.1 Pipeline completo sobre un contrato (Qwen real en GPU)
from rag_core.agent import analyze
contrato = ('Contrato de obra pública por 5M USD con un solo oferente, precios casi '
            'identicos entre los participantes y solo 3 dias entre la publicacion y '
            'la apertura de ofertas.')
result = analyze(contrato, grounding_method='embedding')
print(result['answer'])

## 8. Grounding básico + citación por frase

**Grounding:** dividimos la respuesta en frases y marcamos cada una como *soportada* si supera el umbral de similitud con algún chunk recuperado. El **grounding ratio** = frases soportadas / total. **BONUS citation-per-sentence:** cada frase soportada se cita a su `chunk_id`/indicador/página. Si no hay soporte → **refusal** ('no hay evidencia suficiente').

In [ ]:
# 8.1 Grounding ratio + citas por frase
print('Grounding ratio:', round(result['grounding_ratio'], 3))
for s in result['sentences']:
    print(('OK ' if s['supported'] else '-- '), s['text'][:90])
print('\nCitas:')
for c in result['citations']:
    print('  -', c.get('indicator_code'), c.get('indicator_name'), '| p', c.get('page_start'), '->', c['sentence'][:60])
print('\nRefusal:', result['refusal'] or '(no aplica)')

In [ ]:
# 8.2 Refusal: consulta fuera de dominio -> no inventa
fuera = analyze('Cual es la capital de Francia?', grounding_method='embedding')
print('Respuesta:', fuera['answer'][:200])
print('Refusal:', fuera['refusal'])

## 9. Evaluación

**Métricas:** Recall@k / Precision@k sobre un *gold set* de 12 consultas con indicadores esperados; grounding ratio. **BONUS:** comparación FAISS vs híbrido vs +reranker, y ROUGE/BLEU.

In [ ]:
# 9.1 Recall@k / Precision@k sobre el gold set + comparación de métodos
from evals.metrics import load_goldset, evaluate_on_goldset, compare_methods
from rag_core.retrievers import faiss_search, hybrid_search
from rag_core.rerankers import rerank

def codes(cands):
    out, seen = [], set()
    for c in cands:
        k = c.get('indicator_code')
        if k and k not in seen:
            out.append(k); seen.add(k)
    return out

gold = load_goldset()
faiss_r, hybrid_r, rerank_r = [], [], []
for it in gold:
    qq = it['query']
    faiss_r.append({'retrieved_indicator_codes': codes(faiss_search(qq, k=10, chunks=chunks))})
    hc = hybrid_search(qq, k=10, bm25_obj=bm25, chunks=chunks)
    hybrid_r.append({'retrieved_indicator_codes': codes(hc)})
    pre = hybrid_search(qq, k=20, bm25_obj=bm25, chunks=chunks)
    rerank_r.append({'retrieved_indicator_codes': codes(rerank(qq, pre, top_n=5))})

import json
comp = compare_methods({'faiss': faiss_r, 'hybrid': hybrid_r, 'hybrid+rerank': rerank_r}, gold)
print(json.dumps(comp, indent=2, ensure_ascii=False))

## 10. Análisis de resultados, limitaciones y mejoras

**Qué funciona:** el router clasifica bien la familia; FAISS recupera los indicadores esperados en la mayoría de casos (Recall@5 ~0.83 en muestra local); el grounding + citas reducen alucinación y el sistema **rehúsa** cuando no hay evidencia.

**Caso bueno:** *'planning documents not published'* → recupera R001 (Recall@5 = 1.0).
**Caso malo:** *'single bidder'* → esperaba R018/R019, recuperó R018/R035 (Recall@5 = 0.5): la consulta usa lenguaje distinto al del documento → mejorar con BM25 activo y/o expansión de consulta.

**Limitaciones:** corpus de ~100 pp (un solo PDF); `stage` inferido por heurística; grounding léxico es conservador (en Colab usar `method='embedding'` con Qwen para ratios realistas).

**Mejoras futuras:** GraphRAG de entidades (proveedor/comprador) para red flags relacionales; sub-índices FAISS por familia; comparación de 2 LLMs (Qwen 3B vs 7B).

> ⚠️ **Aviso:** este sistema señala **riesgos potenciales**, no afirma corrupción. Todo resultado **requiere revisión humana**.

In [ ]:
# 10.1 Demo final: contrato con riesgo vs contrato limpio
for nombre, texto in [
    ('CON RIESGO', 'Adjudicacion directa sin licitacion publica, un solo proveedor '
                   'y modificaciones de precio del 60% despues de firmar el contrato.'),
    ('LIMPIO', 'Licitacion abierta con cinco oferentes, criterios publicados y '
              'adjudicacion al menor precio evaluado conforme a las bases.'),
]:
    print('=' * 70, '\n', nombre)
    r = analyze(texto, grounding_method='embedding')
    print(r['answer'][:600])
    print('grounding_ratio:', round(r['grounding_ratio'], 3), '| refusal:', r['refusal'] or '(no)')

In [ ]:
# 10.2 Resumen tecnico final (para la rubrica)
print("===== RESUMEN TECNICO =====")
print("Unidades documentales:", len(units))
print("Chunks:", len(chunks))
print("Embeddings shape:", vectors.shape)
print("FAISS ntotal:", index.ntotal)
print("Modelo embeddings: intfloat/multilingual-e5-base (prefijos query:/passage:)")
print("Modelo generacion: Qwen/Qwen2.5-3B-Instruct")
print("Retrieval: BM25 + FAISS + router de familias")
print("Reranker: BAAI/bge-reranker-v2-m3")
print("Grounding ratio (demo):", round(result.get("grounding_ratio", 0), 3))
print("Aviso etico: senales de riesgo potenciales; requiere revision humana")

## 11. Validación con LangChain (RAG alternativo)

Consultamos/validamos la data con **LangChain**, sobre los **mismos chunks** y el **mismo modelo e5-base** que `rag_core`.

**¿Dónde se guardan los embeddings?**
- `rag_core` (crudo): `data/index/redflags_flatip.index` + `chunk_id_mapping.json`
- **LangChain**: `data/index/langchain_faiss/` (`index.faiss` + `index.pkl`, con textos+metadata) vía `FAISS.save_local`.

Flujo: chunks → `HuggingFaceEmbeddings` → `FAISS.from_documents` → `as_retriever` → `create_retrieval_chain` (con Qwen).

In [ ]:
# 11.0 Dependencias opcionales para validacion con LangChain (BONUS)
RUN_LANGCHAIN = True
try:
    !pip -q install langchain langchain-community langchain-huggingface
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import FAISS as LCFAISS
    print("LangChain OK")
except Exception as e:
    RUN_LANGCHAIN = False
    print("LangChain no disponible -> se omite la seccion 11 (es bonus).")
    print(type(e).__name__, str(e)[:300])

In [ ]:
# 11.1 Validacion con LangChain: SOLO retrieval (sin LLM, sin OOM)
if not RUN_LANGCHAIN:
    print("Seccion 11 omitida: LangChain es bonus opcional.")
else:
    from rag_core.langchain_rag import build_vectorstore, get_retriever, DEFAULT_PERSIST_DIR
    from rag_core.retrievers import load_chunks
    lc_chunks = load_chunks()
    vs = build_vectorstore(lc_chunks)
    print("Embeddings LangChain guardados en:", DEFAULT_PERSIST_DIR)
    for d in get_retriever(vs, k=5).invoke("single bidder with a very short submission period"):
        print("-", d.metadata.get("indicator_code"), d.metadata.get("indicator_name"), "| p", d.metadata.get("page_start"))
        print("   ", d.page_content[:100])

In [ ]:
# 11.2 OPCIONAL PESADO: cadena RAG con Qwen via LangChain (puede duplicar Qwen en memoria)
RUN_LANGCHAIN_QWEN = False  # True solo si tienes RAM/GPU de sobra
if not RUN_LANGCHAIN or not RUN_LANGCHAIN_QWEN:
    print("LangChain+Qwen omitido para no duplicar Qwen en memoria (el Qwen principal ya se uso en la seccion 7).")
else:
    from rag_core.langchain_rag import build_qa_chain, get_qwen_llm
    chain = build_qa_chain(get_retriever(vs, k=5), get_qwen_llm())
    out = chain.invoke({"input": "Un contrato con un solo oferente y 3 dias entre publicacion y apertura. Que senales de riesgo aplican?"})
    print(out["answer"])
    print("\nFuentes citadas:")
    for d in out["context"]:
        print("-", d.metadata.get("indicator_code"), d.metadata.get("indicator_name"), "| p", d.metadata.get("page_start"))

## 12. Chat interactivo — Analizador preliminar de contratos/TDR

Interfaz tipo chatbot (**Gradio**) sobre `rag_core.agent.analyze` (**Qwen** como generador principal). Pega un fragmento de contrato/TDR/licitacion y el sistema devuelve **senales de riesgo potenciales** con criterios, evidencia recuperada (indicadores OCP + pagina), grounding ratio y la nota **"requiere revision humana"**. **No determina corrupcion.**

**Criterios (familias de red flags):** competencia (oferente unico/pocos postores) · plazos cortos · precios identicos/sobrecostos · adjudicacion directa · transparencia (docs faltantes) · modificaciones (adendas) · ejecucion (retrasos) · proveedor recurrente.

In [ ]:
# 12.0 Chat interactivo (Gradio) -> llama a analyze() (Qwen)
!pip -q install gradio
import gradio as gr
from rag_core.agent import analyze

CRITERIOS = ("Competencia (oferente unico/pocos postores), Plazos cortos, Precios identicos/sobrecostos, "
             "Adjudicacion directa, Transparencia (docs faltantes), Modificaciones (adendas), "
             "Ejecucion (retrasos), Proveedor recurrente")

def format_analysis(result):
    md = ["## Analisis preliminar de senales de riesgo", "", result.get("answer", ""),
          "", "**Criterios evaluados:** " + CRITERIOS,
          "", f"**Grounding ratio:** {result.get('grounding_ratio', 0):.2f}"]
    if result.get("refusal"):
        md += ["", "**Aviso:** " + str(result["refusal"])]
    cites = result.get("citations", [])
    if cites:
        md += ["", "**Evidencia recuperada (guia OCP):**"]
        for i, c in enumerate(cites[:5], 1):
            md.append(f"{i}. {c.get('indicator_code','?')} - {c.get('indicator_name','?')} | pag. {c.get('page_start','?')}")
            if c.get("sentence"):
                md.append("   - Frase soportada: " + c["sentence"][:200])
    md += ["", "---", "**Nota:** no determina corrupcion; son senales de riesgo potenciales. **Requiere revision humana.**"]
    return "\n".join(md)

def analizar_contrato_chat(message, history):
    if not message or len(message.strip()) < 20:
        return "Pega un fragmento mas completo del contrato, TDR o licitacion para analizar senales de riesgo."
    try:
        return format_analysis(analyze(message, grounding_method="embedding"))
    except Exception as e:
        return f"Error durante el analisis: {type(e).__name__}: {str(e)[:400]}"

demo = gr.ChatInterface(
    fn=analizar_contrato_chat,
    title="Chat RAG - Analisis preliminar de Red Flags en Contratacion Publica",
    description="Pega un fragmento de contrato/TDR/licitacion. Recupera evidencia de la guia OCP, genera con Qwen y muestra senales de riesgo potenciales. No reemplaza auditoria ni revision legal.",
    examples=[
        "Contrato de obra publica por 5M USD con un solo oferente, precios casi identicos entre participantes y solo 3 dias entre publicacion y apertura de ofertas.",
        "TDR de consultoria que exige experiencia exclusiva con una marca privada, plazo de 48 horas para postular y adjudicacion directa.",
        "Licitacion abierta con cinco oferentes, criterios publicados, evaluacion tecnica documentada y adjudicacion al menor precio.",
    ],
)
demo.launch(debug=False, share=True)

In [ ]:
# 12.1 Escenarios de prueba (sin Gradio) para validar el chat
chat_tests = [
    {"nombre": "Riesgo alto",     "texto": "Un solo oferente, 3 dias de plazo y precios casi identicos."},
    {"nombre": "TDR restrictivo", "texto": "TDR exige experiencia exclusiva con una marca y solo 48 horas para postular."},
    {"nombre": "Caso limpio",     "texto": "Licitacion abierta con cinco oferentes, criterios publicados y evaluacion documentada."},
]
for t in chat_tests:
    r = analyze(t["texto"], grounding_method="embedding")
    print("==", t["nombre"], "| grounding:", round(r.get("grounding_ratio", 0), 2))
    print(r["answer"][:300]); print("-" * 60)

### 12.2 (OPCIONAL / BONUS) Segunda opinion con MiniMax

Qwen es el generador **obligatorio**. MiniMax queda como *segunda opinion* opcional (no rompe la demo). En Secrets configura `MINIMAX_API_KEY` (y opcional `MINIMAX_BASE_URL` / `MINIMAX_MODEL` segun tu plan) y pon `RUN_MINIMAX = True`.

In [ ]:
# 12.2 (opcional) MiniMax como segunda opinion via LangChain (no es el principal)
RUN_MINIMAX = False  # True solo si configuraste MINIMAX_API_KEY en Secrets
if not RUN_MINIMAX:
    print("MiniMax omitido (opcional). Qwen es el generador principal exigido por la rubrica.")
else:
    !pip -q install langchain langchain-openai langchain-community langchain-huggingface
    from rag_core.langchain_rag import build_qa_chain, get_minimax_llm, build_vectorstore, get_retriever
    from rag_core.retrievers import load_chunks
    vs_mm = build_vectorstore(load_chunks())
    chain = build_qa_chain(get_retriever(vs_mm, k=5), get_minimax_llm())
    out = chain.invoke({"input": "Un contrato con un solo oferente y 3 dias entre publicacion y apertura. Que senales de riesgo aplican?"})
    print(out["answer"])